# Satellite Imagery Data Ingestion

Loads SkySat image chips and their whale/boat label masks straight from Google Drive and
wraps them in a PyTorch `Dataset`. Meant to run in Colab.

**Drive layout** (each label file has the same name as the image it belongs to):

```
images/000000000035.tif          500x500, 4 bands, uint16
labels/boats/000000000035.tif    500x500, 1 band, uint16 mask
labels/whales/000000000447.tif
```

Each `.tif` comes with `.tfw` and `.tif.aux.xml` sidecar files. Those hold georeferencing
and statistics metadata, and this notebook doesn't need them.

## Setup

In [ ]:
!pip install -q rasterio

# Colab caches site-packages at startup, so make freshly installed packages importable.
import importlib
import site

importlib.reload(site)

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from torch.utils.data import DataLoader, Dataset

## Google Drive helpers

In [ ]:
auth.authenticate_user()
svc = build('drive', 'v3')


def download_file(file_id):
    """Download a Drive file into an in-memory file object."""
    request = svc.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)
    return fh


def ls(folder_id, n=10):
    """Print the first n files in a Drive folder as name | type | size | id."""
    resp = (
        svc.files()
        .list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='files(id, name, mimeType, size)',
            pageSize=n,
        )
        .execute()
    )
    for f in resp.get('files', []):
        print(f'{f["name"]} | {f["mimeType"]} | {f.get("size", "-")} | {f["id"]}')

## Look at the label folders

In [ ]:
WHALES_DIR_ID = '1VgVAk2T-9HyM7ZSzbfCrkaJoHdAkzDWI'
BOATS_DIR_ID = '1SUqFbNZFPYXLeZo4Q3J6Qsgn9Ea791ho'

print("Files inside 'labels/whales':")
ls(WHALES_DIR_ID, n=5)
print("\nFiles inside 'labels/boats':")
ls(BOATS_DIR_ID, n=5)

## Build the full file list

List every `.tif` in the image folder and both label folders, then match them up by filename.
A chip can have a boat mask, a whale mask, both, or neither. A missing mask is stored as
`None`, so every image stays in the list, including empty chips.

In [ ]:
# The image folder is whichever folder holds the known sample image (000000000035.tif).
IMAGES_DIR_ID = (
    svc.files().get(fileId='1_USaVdJJBScGz1dngPkiIAtY2hBQ2EK9', fields='parents').execute()
)['parents'][0]


def list_tifs(folder_id):
    """Return {filename: file_id} for every .tif in a Drive folder (skips .tfw/.aux.xml)."""
    files = {}
    page_token = None
    while True:
        resp = (
            svc.files()
            .list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields='nextPageToken, files(id, name)',
                pageSize=1000,
                pageToken=page_token,
            )
            .execute()
        )
        for f in resp.get('files', []):
            if f['name'].endswith('.tif'):
                files[f['name']] = f['id']
        page_token = resp.get('nextPageToken')
        if page_token is None:
            return files


image_files = list_tifs(IMAGES_DIR_ID)
whale_files = list_tifs(WHALES_DIR_ID)
boat_files = list_tifs(BOATS_DIR_ID)

records = [
    {
        'name': name,
        'image_id': image_id,
        'whale_id': whale_files.get(name),
        'boat_id': boat_files.get(name),
    }
    for name, image_id in sorted(image_files.items())
]

n_whale = sum(r['whale_id'] is not None for r in records)
n_boat = sum(r['boat_id'] is not None for r in records)
n_empty = sum(r['whale_id'] is None and r['boat_id'] is None for r in records)
print(
    f'{len(records)} images | {n_whale} with whale mask | {n_boat} with boat mask | '
    f'{n_empty} with neither'
)

# Label files whose image is missing usually mean a naming mismatch.
orphans = (set(whale_files) | set(boat_files)) - set(image_files)
print(f'{len(orphans)} label files with no matching image', sorted(orphans)[:5])
records[:3]

## Download everything to local disk

Saves every image and mask to `data/` once. Files that are already there get skipped, so
it's safe to re-run.

In [ ]:
from pathlib import Path

DATA_DIR = Path('data')  # on Colab this is /content/data

for subdir, files in [
    ('images', image_files),
    ('labels/whales', whale_files),
    ('labels/boats', boat_files),
]:
    out_dir = DATA_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, (name, file_id) in enumerate(sorted(files.items()), 1):
        out_path = out_dir / name
        if not out_path.exists():
            out_path.write_bytes(download_file(file_id).getvalue())
        if i % 100 == 0 or i == len(files):
            print(f'{subdir}: {i}/{len(files)}')

## Inspect one image/mask pair

`rasterio` reads a GeoTIFF as a `(bands, height, width)` array. The printout below shows the
file's metadata: band count, dtype, compression, and band descriptions if the file has any.

In [ ]:
sample_image_id = '1_USaVdJJBScGz1dngPkiIAtY2hBQ2EK9'  # images/000000000035.tif
sample_boat_label_id = '1ydT8-SNdYvcw1djISFA3agkOMWCYNN8Z'  # labels/boats/000000000035.tif

with rasterio.open(download_file(sample_image_id)) as src:
    print('Image profile:', src.profile)
    print('Band descriptions:', src.descriptions)
    img_np = src.read()  # (bands, H, W)

with rasterio.open(download_file(sample_boat_label_id)) as src:
    mask_np = src.read(1)  # first (only) band -> (H, W)

print(f'\nImage shape: {img_np.shape} | dtype: {img_np.dtype}')
print(f'Mask shape : {mask_np.shape} | dtype: {mask_np.dtype}')
print(f'Unique values in mask: {np.unique(mask_np)}')
for b, band in enumerate(img_np):
    print(f'Band {b}: min={band.min()}, max={band.max()}, mean={band.mean():.1f}')

SkySat analytic products usually store bands as **Blue, Green, Red, NIR**, so the display
below uses bands `[2, 1, 0]` to get true color. If the band descriptions above say otherwise,
change `RGB_BANDS`.

In [ ]:
RGB_BANDS = [2, 1, 0]

img_vis = np.moveaxis(img_np[RGB_BANDS], 0, -1).astype(np.float32)  # (H, W, 3)
img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min() + 1e-5)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_vis)
axes[0].set_title('Input image (000000000035.tif)')
axes[0].axis('off')

axes[1].imshow(mask_np, cmap='gray')
axes[1].set_title('Boat mask')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## PyTorch Dataset

Downloads each chip on demand, scales it to `[0, 1]`, and returns a `(C, H, W)` float tensor
(plus an `(H, W)` mask tensor when labels are given). `transform` follows the Albumentations
call style, `transform(image=..., mask=...)`, which expects `(H, W, C)` arrays. That's why the
image moves its bands to the last axis and then back again.

In [ ]:
class SatelliteDetectionDataset(Dataset):
    def __init__(self, image_ids, label_ids=None, transform=None):
        """
        Args:
            image_ids (list of str): Google Drive file IDs for the input images.
            label_ids (list of str, optional): Drive file IDs for the matching masks.
            transform (callable, optional): Albumentations-style transform.
        """
        self.image_ids = image_ids
        self.label_ids = label_ids
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        with rasterio.open(download_file(self.image_ids[idx])) as src:
            img_np = src.read().astype(np.float32)  # (C, H, W)

        img_np = np.moveaxis(img_np, 0, -1)  # (H, W, C)
        # Per-chip min-max over all bands at once (placeholder; see DATA.md on a fixed stretch).
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-5)

        if self.label_ids is None:
            if self.transform:
                img_np = self.transform(image=img_np)['image']
            return torch.from_numpy(img_np).permute(2, 0, 1)  # (C, H, W)

        with rasterio.open(download_file(self.label_ids[idx])) as src:
            mask_np = src.read(1).astype(np.int64)  # (H, W)

        if self.transform:
            augmented = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = augmented['image'], augmented['mask']

        return torch.from_numpy(img_np).permute(2, 0, 1), torch.from_numpy(mask_np)

## Test with a DataLoader

In [ ]:
dataset = SatelliteDetectionDataset([sample_image_id], [sample_boat_label_id])
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

for batch_idx, (images, masks) in enumerate(dataloader):
    print(f'Batch {batch_idx + 1}:')
    print(f'  images: {images.shape} {images.dtype}')  # (B, 4, 500, 500)
    print(f'  masks : {masks.shape} {masks.dtype}')  # (B, 500, 500)